# CV Field Mapper — LoRA Fine-tune (Qwen2.5-1.5B)

**Settings:** GPU (A100 nếu có) · Persistence: Files only · Internet **ON** (tải model HF).

Cách dùng:
1. Add Data → Upload zip repo `cv-mapper-finetune` **hoặc** clone GitHub ở cell dưới.
2. Run All.
3. Download `/kaggle/working/lora` về máy.

In [ ]:
import os, torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('bf16:', torch.cuda.is_bf16_supported())

## Option A — clone từ GitHub (sửa URL của bạn)

In [ ]:
# Sửa URL sau khi push repo
REPO = os.environ.get('CV_MAPPER_REPO', 'https://github.com/YOUR_USER/cv-mapper-finetune.git')
WORK = '/kaggle/working/cv-mapper-finetune'
if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
%cd {WORK}
!ls -la

## Option B — nếu đã Add Data (Kaggle Dataset)
Uncomment và sửa path dataset.

In [ ]:
# import shutil
# SRC = '/kaggle/input/cv-mapper-finetune'  # tên dataset trên Kaggle
# WORK = '/kaggle/working/cv-mapper-finetune'
# if not os.path.exists(WORK):
#     shutil.copytree(SRC, WORK)
# %cd {WORK}

In [ ]:
!pip install -q -r requirements.txt
from pathlib import Path
train = Path('data/train.jsonl')
if not train.exists():
    !python generate_dataset.py --n 1600
print('train lines:', sum(1 for _ in open('data/train.jsonl', encoding='utf-8')))

In [ ]:
# A100 defaults: bf16 auto, batch 4, seq 4096, 2 epochs
!python train_lora.py \
  --out /kaggle/working/lora \
  --epochs 2 \
  --batch-size 4 \
  --grad-accum 4 \
  --max-seq-length 4096 \
  --save-steps 100 \
  --logging-steps 10

In [ ]:
!ls -lah /kaggle/working/lora
!cat /kaggle/working/lora/train_meta.json
print('Done — download /kaggle/working/lora from Output')